<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Dailychallenge_Mini8projet_W8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BaristaBot: Building a Stateful Agent with LangGraph and Gemini

This notebook guides you through building a conversational cafe ordering system. We will use **LangGraph** for state management and **Gemini** for the intelligence.

### 1. Install Dependencies

In [ ]:
%pip install -qU "langgraph==0.2.60" "langchain-google-genai==2.0.8" "google-genai==1.1.0"

### 2. Setup API Key
We'll use Colab's secret manager to securely handle the API key.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    print("Please add your GOOGLE_API_KEY to Colab Secrets.")

### 3. Define State and Instructions
LangGraph uses a `TypedDict` to track the state of the conversation (messages, current order, and completion status).

In [ ]:
from typing import Annotated, Literal, Iterable
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool

BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. "
    "Answer menu questions and manage orders. "
    "Add items with add_to_order, reset with clear_order, see contents with get_order. "
    "Always confirm_order with the user before calling place_order. "
    "Once place_order returns, thank the user and say goodbye!"
)

WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"

### 4. Define Tools and Nodes
We define the tools the bot can use (like getting the menu) and the nodes (steps) in our graph.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage, ToolMessage
from random import randint

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    MENU:
    Coffee: Espresso, Americano, Cold Brew, Latte, Cappuccino, Mocha.
    Tea: English Breakfast, Green Tea, Chai Latte, Matcha Latte.
    Modifiers: Milk (Whole, 2%, Oat, Almond), Shots (Single, Double), Caffeine (Decaf, Regular).
    """

@tool
def add_to_order(drink: str, modifiers: Iterable[str]):
    """Adds the specified drink and modifiers to the order."""

@tool
def confirm_order():
    """Asks the customer if the order is correct."""

@tool
def get_order():
    """Returns the users order so far."""

@tool
def clear_order():
    """Removes all items from the order."""

@tool
def place_order():
    """Sends the order to the kitchen."""

# Grouping for logic
auto_tools = [get_menu]
order_tool_names = ["add_to_order", "confirm_order", "get_order", "clear_order", "place_order"]
all_tools = auto_tools + [add_to_order, confirm_order, get_order, clear_order, place_order]
llm_with_tools = llm.bind_tools(all_tools)

def chatbot(state: OrderState):
    if not state.get("messages"):
        return {"messages": [AIMessage(content=WELCOME_MSG)], "order": [], "finished": False}
    response = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    return {"messages": [response]}

def human_node(state: OrderState):
    last_msg = state["messages"][-1]
    print(f"\nBot: {last_msg.content}")
    user_input = input("User: ")
    if user_input.lower() in ["q", "quit", "exit"]:
        return {"messages": [("user", user_input)], "finished": True}
    return {"messages": [("user", user_input)]}

def order_node(state: OrderState):
    tool_msg = state["messages"][-1]
    order = state.get("order", [])
    outbound_msgs = []
    finished = False

    for tool_call in tool_msg.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]
        res = "Done"

        if name == "add_to_order":
            item = f"{args['drink']} ({', '.join(args.get('modifiers', []))})"
            order.append(item)
            res = f"Added {item} to order."
        elif name == "confirm_order":
            print("\n--- Current Order ---")
            for i in order: print(f"- {i}")
            res = input("Is this correct? ")
        elif name == "get_order":
            res = "\n".join(order) if order else "Your order is empty."
        elif name == "clear_order":
            order = []
            res = "Order cleared."
        elif name == "place_order":
            print(f"\nKitchen: Receiving order... {order}")
            finished = True
            res = f"Order placed! ETA: {randint(5,15)} mins."

        outbound_msgs.append(ToolMessage(content=str(res), tool_call_id=tool_call["id"]))

    return {"messages": outbound_msgs, "order": order, "finished": finished}

### 5. Build the Graph
Now we connect the nodes into a workflow.

In [ ]:
from langgraph.graph import StateGraph, START, END

def router(state: OrderState):
    if state.get("finished"):
        return END
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        if any(tc["name"] in order_tool_names for tc in last_msg.tool_calls):
            return "ordering"
        return "tools"
    return "human"

builder = StateGraph(OrderState)
builder.add_node("chatbot", chatbot)
builder.add_node("human", human_node)
builder.add_node("tools", ToolNode(auto_tools))
builder.add_node("ordering", order_node)

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", router)
builder.add_edge("tools", "chatbot")
builder.add_edge("ordering", "chatbot")
builder.add_conditional_edges("human", lambda s: END if s.get("finished") else "chatbot")

app = builder.compile()

### 6. Run the Bot
You can now interact with the compiled graph.

In [ ]:
print("Starting BaristaBot. Type 'q' to exit.")
# Start the conversation
app.invoke({"messages": [], "order": [], "finished": False}, {"recursion_limit": 100})